In [1]:
import os
%env CUDA_VISIBLE_DEVICES=1
os.environ["CUDA_VISIBLE_DEVICES"]="1"
from transformers import LlamaForCausalLM, AutoTokenizer
import torch
import json

env: CUDA_VISIBLE_DEVICES=1


/data2/polyakov/anaconda3/envs/alpaca-lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tok = AutoTokenizer.from_pretrained('/data2/khrapov/Llama-2-13B-fp16/')
model = LlamaForCausalLM.from_pretrained(
        '/data2/khrapov/Llama-2-13B-fp16/',
        torch_dtype=torch.float16,
        local_files_only=True,
        device_map='auto',
)

Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████| 3/3 [00:20<00:00,  6.95s/it]


In [3]:
def generate(input, max_len=30):
    tokenized = tok(input, return_tensors="pt").input_ids.to('cuda:0')
    res = model.generate(tokenized, max_length=30)
    return tok.batch_decode(res, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

In [4]:
generate('Who are you?')

'Who are you? What is your name?\nI am a 20 year old student from the Netherlands. I am currently studying at the'

Примеры ошибок

In [12]:
errors = json.load(open('responses_message_error.json', 'r'))
query = errors[0]['answer_generation']['query']
error_chain_example = errors[0]['trys'][0]['chain'][:3]
tool_descriptions = str(errors[0]['answer_generation']['function'])
errors[0]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'get_distance_by_city_state_country_for_great_circle_math_api',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "country1": "US",\n  "country2": "US",\n  "state2": "CA",\n  "city2": "San Bernardino",\n  "city1": "Norwalk",\n  "state1": "AK"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "", "response": "{\'cityName1\': \'Incorrect city/state/country\', \'stateName1\': \'Incorrect city/state/country\', \'country1\': \'Incorr

In [13]:
prompt = ''
for item in error_chain_example:
    if item['node_type'] != 'Action Input':
        prompt += item['node_type'] + ': ' + item['description'] + '\n'
    else:
        prompt += item['node_type'] + ': ' + item['description'] + '\n' + 'Observation: ' + item['observation'] + '\n'

In [17]:
request = f'The user asked: {query}\n'
tool_descriptions_prompt = f'Here are tool descriptions: {tool_descriptions}\n'
initial_prompt = 'You are given an example of tool call by a model called Toollama and a response from a tool.\nExample begins\n'
final_prompt = tool_descriptions_prompt + request + initial_prompt + prompt + 'Example ends\nAnswer, if this call resulted in an error. If so, explain an error. Do not write any code. Do not make up anything. Explain an error based on tool description, tool parameters format (in the field "parameters"), on response from tool and on your knowledge.\n'
print(final_prompt)

Here are tool descriptions: [{'name': 'get_distance_by_city_state_country_for_great_circle_math_api', 'description': 'This is the subfunction for tool "great_circle_math_api", you can use this tool.The description of this function is: "Takes city, state, and country of both locations and returns latitude, longitude, and calculated miles."', 'parameters': {'type': 'object', 'properties': {'country1': {'type': 'string', 'description': '', 'example_value': 'us'}, 'country2': {'type': 'string', 'description': '', 'example_value': 'us'}, 'state2': {'type': 'string', 'description': '', 'example_value': 'ca'}, 'city2': {'type': 'string', 'description': '', 'example_value': 'sacramento'}, 'city1': {'type': 'string', 'description': '', 'example_value': 'birmingham'}, 'state1': {'type': 'string', 'description': '', 'example_value': 'al'}}, 'required': ['country1', 'country2', 'state2', 'city2', 'city1', 'state1'], 'optional': []}}, {'name': 'm_one_unit_of_measure_to_another_for_measurement_units

In [18]:
# res = generate(final_prompt)
tokenized = tok(final_prompt, return_tensors="pt").input_ids.to('cuda:0')
res = model.generate(tokenized, max_length=tokenized.shape[1] + 200)
# return tok.batch_decode(res, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

In [19]:
print(tok.batch_decode(res)[0])

<s> Here are tool descriptions: [{'name': 'get_distance_by_city_state_country_for_great_circle_math_api', 'description': 'This is the subfunction for tool "great_circle_math_api", you can use this tool.The description of this function is: "Takes city, state, and country of both locations and returns latitude, longitude, and calculated miles."', 'parameters': {'type': 'object', 'properties': {'country1': {'type': 'string', 'description': '', 'example_value': 'us'}, 'country2': {'type': 'string', 'description': '', 'example_value': 'us'}, 'state2': {'type': 'string', 'description': '', 'example_value': 'ca'}, 'city2': {'type': 'string', 'description': '', 'example_value': 'sacramento'}, 'city1': {'type': 'string', 'description': '', 'example_value': 'birmingham'}, 'state1': {'type': 'string', 'description': '', 'example_value': 'al'}}, 'required': ['country1', 'country2', 'state2', 'city2', 'city1', 'state1'], 'optional': []}}, {'name': 'm_one_unit_of_measure_to_another_for_measurement_u

In [34]:
tokenized

tensor([[    1,   887,   526,  2183,   385,  1342,   310,  5780,  1246,   491,
          1904,  2000, 21704, 29880,  3304, 29889,    13, 14023, 16410,    13,
          1349,  1774, 29901, 29871,    13,  4276, 29901,   679, 29918, 19244,
         29918,  1609, 29918, 12690, 29918,  3859, 29918, 13509, 29918,  1454,
         29918,  7979,   271, 29918, 16622, 29918,   755, 29918,  2754,    13,
          4276, 10567, 29901,   426,    13, 29871,   376, 13509, 29896,  1115,
           376,  3308,   613,    13, 29871,   376, 13509, 29906,  1115,   376,
          3308,   613,    13, 29871,   376,  3859, 29906,  1115,   376,  5454,
           613,    13, 29871,   376, 12690, 29906,  1115,   376, 22509, 13337,
          1789,   613,    13, 29871,   376, 12690, 29896,  1115,   376, 29940,
           272, 20919,   613,    13, 29871,   376,  3859, 29896,  1115,   376,
         22311, 29908,    13, 29913,    13,  6039,  2140,   362, 29901,  8853,
          2704,  1115, 12633,   376,  5327,  1115,  

In [29]:
print(res)

You are given an example of tool call by model called Toollama.
Example begins
Thought: 
Action: get_distance_by_city_state_country_for_great_circle_math_api
Action Input: {
  "country1": "US",
  "country2": "US",
  "state2": "CA",
  "city2": "San Bernardino",
  "city1": "Norwalk",
  "state1": "AK"
}
Observation: {"error": "", "response": "{'cityName1': 'Incorrect city/state/country', 'stateName1': 'Incorrect city/state/country', 'country1': 'Incorrect city/state/country', 'latitude1': 'NaN', 'longitude1': 'NaN', 'cityName2': 'San Bernardino', 'stateName2': 'California', 'country2': 'US', 'latitude2': 34.1083449, 'longitude2': -117.2897652, 'calculatedDist': {'distance': 0.0, 'uom': 'mi'}}"}
Example ends
If this call resulted in an error, please describe it.
Begin!
































































































































































































































